In [14]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [22]:
train_datagen = ImageDataGenerator(
    rescale=1./255,       
    rotation_range=20,    
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))


test_dir = os.path.join(BASE_DIR, "data", "preprocessed_split_2","test")
train_dir = os.path.join(BASE_DIR, "data", "preprocessed_split_2","train")
val_dir = os.path.join(BASE_DIR, "data", "preprocessed_split_2","val")

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),   
    batch_size=16,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary',
)




Found 2315 images belonging to 2 classes.
Found 661 images belonging to 2 classes.
Found 333 images belonging to 2 classes.


In [28]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


In [29]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x) 

model = Model(inputs=base_model.input, outputs=predictions)


In [30]:
for layer in base_model.layers:
    layer.trainable = True


In [31]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)


In [32]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop,lr_reduce]
)


Epoch 1/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 510s 3s/step - accuracy: 0.8968 - loss: 0.2887 - val_accuracy: 0.5113 - val_loss: 432.1335 - learning_rate: 0.0010
Epoch 2/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 468s 3s/step - accuracy: 0.9296 - loss: 0.1864 - val_accuracy: 0.4841 - val_loss: 0.7058 - learning_rate: 0.0010
Epoch 3/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 6767s 47s/step - accuracy: 0.9387 - loss: 0.1622 - val_accuracy: 0.4887 - val_loss: 0.8120 - learning_rate: 0.0010
Epoch 4/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 15022s 104s/step - accuracy: 0.9508 - loss: 0.1425 - val_accuracy: 0.4932 - val_loss: 0.8221 - learning_rate: 0.0010
Epoch 5/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 3572s 25s/step - accuracy: 0.9499 - loss: 0.1433 - val_accuracy: 0.5113 - val_loss: 7.6982 - learning_rate: 0.0010
Epoch 6/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 2587s 18s/step - accuracy: 0.9559 - loss: 0.1307 - val_accuracy: 0.5113 - val_loss: 6.6310 - learning_rate: 0.0010
Epoch 7/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 1541s 11s/step - accuracy: 0.9

In [34]:
import json

# Save history
with open("resnet.json", "w") as f:
    json.dump(history.history, f)


In [ ]:
import matplotlib.pyplot as plt

# -------- LOSS --------
plt.figure()
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# -------- ACCURACY --------
plt.figure()
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()


In [ ]:
y_true = test_generator.classes
y_pred_probs = model.predict(test_generator, verbose=1)

y_pred = np.argmax(y_pred_probs, axis=1)

test_acc = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred)
test_recall = recall_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred)

print("Test Accuracy :", test_acc)
print("Test Precision:", test_precision)
print("Test Recall   :", test_recall)
print("Test F1-Score :", test_f1)

In [33]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
model_dir = os.path.join(BASE_DIR, "models")

os.makedirs(model_dir, exist_ok=True)

model.save(os.path.join(model_dir, "ResNet50_trained.keras"))

In [ ]:
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(train_generator, validation_data=val_generator, epochs=10)
